In [1]:
import requests
import urllib.parse as up
import pandas as pd
from pandas import json_normalize
import json
import glob
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
from sklearn.linear_model import LinearRegression
from dash import Dash, html, dcc, callback, Output, Input
import plotly.express as px
import dash_ag_grid as dag
import plotly.graph_objects as go

In [19]:
df_d_PM10 = pd.read_csv("neujahr_PM10.csv")
df_d_PM25 = pd.read_csv("neujahr_PM2,5.csv")
df_d_NO2 = pd.read_csv("neujahr_NO2.csv")

data = {
    "daily avg PM<sub>10</sub>": df_d_PM10,
    "daily avg PM<sub>2.5</sub>": df_d_PM25,
    "daily avg NO<sub>2</sub>": df_d_NO2,
}

for df in data.values():
    df["date start"] = pd.to_datetime(df["date start"])

app = Dash(__name__)

app.layout = html.Div([
    html.H1("Air Pollution Dashboard Germany"),
    dcc.Graph(id="pollution-plot")
])


@app.callback(
    Output("pollution-plot", "figure"),
    Input("pollution-plot", "relayoutData")
)
def update_plot(relayoutData):

    fig = go.Figure()

    for name, df in data.items():

        filtered = df

        if relayoutData and "xaxis.range[0]" in relayoutData:

            start = relayoutData["xaxis.range[0]"]
            end = relayoutData["xaxis.range[1]"]

            filtered = df[
                (df["date start"] >= start) &
                (df["date start"] <= end)
            ]

        fig.add_trace(
            go.Scatter(
                x=filtered["date start"],
                y=filtered["value"],
                mode="markers",
                name=name
            )
        )

        if len(filtered) > 2:

            x_num = filtered["date start"].map(pd.Timestamp.toordinal)
            y = filtered["value"]

            coeff = np.polyfit(x_num, y, 2)
            poly = np.poly1d(coeff)

            fig.add_trace(
                go.Scatter(
                    x=filtered["date start"],
                    y=poly(x_num),
                    mode="lines",
                    name=name + " trend"
                )
            )

    fig.update_layout(
        title="Air Pollution Germany",
        xaxis_title="Date",
        yaxis_title="Concentration in µg/m³",
        hovermode="closest"
    )

    return fig


if __name__ == "__main__":
    app.run(debug=True)

In [21]:
#fig.write_html("question_1.2.html")